# 01 - Setup Bigquery
Creación del dataset y las tablas para INNOVACIBER en Google Bigquery

**Orden de creación** (respetando dependencias FK):
1. `categories`
2. `customers`
3. `products` (depende de categories)
4. `orders` (depende de customers)
5. `order_items` (depende de orders y productos)
6. `payments` (depende de orders)
7. `shipments` (depende de orders)
8. `reviews` (depende de customers y products)

## 1. Importaciones y configuración

In [9]:
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv

load_dotenv()


True

In [18]:

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

## 2. Autentificación cliente

In [19]:
credentials = service_account.Credentials.from_service_account_file(CREDENTIALS_PATH)

client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

## 3. Crear el dataset

In [12]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"
dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset {DATASET_ID} creado")

Dataset datalearn_academy creado


## 4. Definición de esquemas

In [17]:
F = bigquery.SchemaField # alias corto

SCHEMAS = {

    "categories": [
        F("category_id",  "INTEGER",  mode="REQUIRED", description="PK"),
        F("name",         "STRING",   mode="REQUIRED", description="Nombre de la categoría"),
        F("description",  "STRING",   mode="NULLABLE", description="Descripción"),
    ],

    "customers": [
        F("customer_id",          "INTEGER",   mode="REQUIRED", description="PK"),
        F("first_name",           "STRING",    mode="REQUIRED", description="Nombre"),
        F("last_name",            "STRING",    mode="REQUIRED", description="Apellido"),
        F("email",                "STRING",    mode="REQUIRED", description="Email único"),
        F("country",              "STRING",    mode="REQUIRED", description="País (ISO 3166-1 alpha-2)"),
        F("city",                 "STRING",    mode="NULLABLE", description="Ciudad"),
        F("adquisition_channel",  "STRING",    mode="NULLABLE", description="organic | paid_ads | social | referral"),
        F("registration_date",    "TIMESTAMP", mode="REQUIRED", description="Fecha de registro"),
    ],

    "products": [
        F("product_id",   "INTEGER",  mode="REQUIRED", description="PK"),
        F("category_id",  "INTEGER",  mode="REQUIRED", description="FK → categories.category_id"),
        F("product_name", "STRING",   mode="REQUIRED", description="Nombre del producto"),
        F("brand",        "STRING",   mode="NULLABLE", description="Marca comercial del producto"),
        F("sale_price",   "NUMERIC",  mode="REQUIRED", description="Precio de venta (€)"),
        F("cost_price",   "NUMERIC",  mode="REQUIRED", description="Coste de adquisición (€)"),
        F("stock",        "INTEGER",  mode="REQUIRED", description="Unidades en stock"),
        F("is_active",    "BOOLEAN",  mode="REQUIRED", description="Producto visible en catálogo"),
    ],

    "orders": [
        F("order_id",          "INTEGER",   mode="REQUIRED", description="PK"),
        F("customer_id",       "INTEGER",   mode="REQUIRED", description="FK → customers.customer_id"),
        F("order_date",        "TIMESTAMP", mode="REQUIRED", description="Fecha del pedido"),
        F("order_status",      "STRING",    mode="REQUIRED", description="pending | confirmed | shipped | delivered | cancelled | returned"),
    ],

    "order_items": [
        F("order_item_id","INTEGER",  mode="REQUIRED", description="PK"),
        F("order_id",     "INTEGER",  mode="REQUIRED", description="FK → orders.order_id"),
        F("product_id",   "INTEGER",  mode="REQUIRED", description="FK → products.product_id"),
        F("quantity",     "INTEGER",  mode="REQUIRED", description="Unidades compradas"),
        F("unit_price",   "NUMERIC",  mode="REQUIRED", description="Precio en el momento de compra"),
    ],

    "payments": [
        F("payment_id",    "INTEGER",   mode="REQUIRED", description="PK"),
        F("order_id",      "INTEGER",   mode="REQUIRED", description="FK → orders.order_id"),
        F("payment_method","STRING",    mode="REQUIRED", description="credit_card | paypal | bank_transfer"),
        F("payment_status","STRING",    mode="REQUIRED", description="completed | refunded | pending | failed"), 
        F("payment_date",  "TIMESTAMP", mode="REQUIRED", description="Fecha del pago"),
        F("amount",        "NUMERIC",   mode="REQUIRED", description="Importe total pagado (€)"),
    ],

    "shipments": [
        F("shipments_id",    "INTEGER",   mode="REQUIRED", description="PK"),
        F("order_id",        "INTEGER",   mode="REQUIRED", description="FK → payments.order_id"),
        F("shipping_status", "STRING",    mode="REQUIRED", description="pending | processing | shipped | in_transit | delivered | returned | cancelled"),
        F("shipping_country","STRING",    mode="REQUIRED", description="País (ISO 3166-1 alpha-2)"),
        F("shipping_city",   "STRING",    mode="REQUIRED", description="Ciudad"),
        F("delivery_date",   "TIMESTAMP", mode="REQUIRED", description="Fecha de entrega"),
        F("street_address",  "STRING",    mode="REQUIRED", description="Dirección con calle, numero y piso"),
        F("cp_code",         "STRING",    mode="REQUIRED", description="Código postal"),
        F("phone",           "STRING",    mode="REQUIRED", description="Ciudad"),
    ],

    "reviews": [
        F("review_id",   "INTEGER",   mode="REQUIRED", description="PK"),
        F("customer_id", "INTEGER",   mode="REQUIRED", description="FK → customers.customer_id"),
        F("product_id",   "INTEGER",   mode="REQUIRED", description="FK → products.product_id "),
        F("rating",      "INTEGER",   mode="REQUIRED", description="Valoración 1-5"),
        F("review_date", "TIMESTAMP", mode="REQUIRED", description="Fecha de la reseña"),
        F("comment",     "STRING",    mode="NULLABLE", description="Comentario del cliente"),
        
    ],
}

print(f"{len(SCHEMAS)} esquemas definidos: {list(SCHEMAS.keys())}")

8 esquemas definidos: ['categories', 'customers', 'products', 'orders', 'order_items', 'payments', 'shipments', 'reviews']


## 5. Creación de tablas

In [15]:
TABLAS = [
    "categories",
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "reviews",
]

In [ ]:
for tabla in TABLAS:
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{tabla}"
    table = bigquery.Table(table_ref, schema=SCHEMAS[tabla])
    table = client.create_table(table, exists_ok=True)
    print(f"Tabla {tabla} creada")
    print(table.table_id)

Tabla categories creada
categories
Tabla customers creada
customers
Tabla products creada
products
Tabla orders creada
orders
Tabla order_items creada
order_items
Tabla payments creada
payments
Tabla shipments creada
shipments
Tabla reviews creada
reviews


## 6. (Opcional) Eliminar todas las tablas para empezar de cero

In [ ]:
# PELIGRO: esto borra todos los datos, lo dejo comentado para que no se ejecute por error.

# tablas_elim = list(reversed(tablas))  # inverso para respetar FKs
# for tabla in tablas_elim:
#    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{tabla}"
#    client.delete_table(table_ref, not_found_ok=True)
 #   print(f"{tabla} eliminada")

reviews eliminada
shipments eliminada
payments eliminada
order_items eliminada
orders eliminada
products eliminada
customers eliminada
categories eliminada
